# Часть 1. Данные и EDA

Этот ноутбук скачивает датасеты и готовит файлы для остальных частей. Повторный запуск не качает архивы заново.

Перед первым запуском в корне репозитория:

```powershell
python -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
```

Полный прогон забирает около 7 ГБ и просит примерно 15 ГБ свободного места. Флаг `QUICK = True` ниже проверяет только `test.zip`; на нём обучать модели нельзя.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "prepare_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# False — полный датасет для следующих частей.
# True — только test.zip, чтобы проверить, что ноутбук запускается.
QUICK = False

from prepare_data import prepare

summary = prepare(ROOT, quick=QUICK)


## Что получилось

Ниже выводы, размеры выборок и графики. Их же можно открыть из `reports/part1_summary.json`, если ноутбук уже выполнялся.

Дальше другим людям не нужно распаковывать архивы:

- детекция читает `configs/plates.yaml`;
- OCR читает `data/processed/ocr/train.csv` (колонки `image` и `text`).


In [ ]:
import json

import pandas as pd
from IPython.display import Image, Markdown, display

summary_file = ROOT / "reports" / "part1_summary.json"
if not summary_file.exists():
    raise FileNotFoundError("Сначала выполните ячейку подготовки выше.")
summary = json.loads(summary_file.read_text(encoding="utf-8"))

display(Markdown("\n".join(f"- {line}" for line in summary["conclusions"])))
display(Markdown("### Детекция по выборкам"))
display(pd.DataFrame(summary["tables"]["detection_splits"]))
display(Markdown("### OCR по выборкам"))
display(pd.DataFrame(summary["tables"]["ocr_splits"]))

for plot in summary["plots"]:
    display(Markdown(f"### {plot['title']}"))
    display(Image(filename=plot["path"]))
    display(Markdown(plot["reading"]))

print("YAML для YOLO:", summary["outputs"]["detection_yaml"])
print("CSV для OCR:", summary["outputs"]["ocr_csv"])
